# Image Moderation — TF NSFW classifier (replaces NudeNet baseline)

Trains a binary NSFW classifier on the local corpus
(`data/nsfw/out/{train,val,test}/{Neutral,NSFW}`), plus the Reddit
title->is_nsfw table as a text prior. Serves `app/moderation_engine.py`
`/api/v1/moderation/image` (currently NudeNet + pixel-analysis fallback); this
model is the trained replacement. Exported ONNX takes a 224x224x3 float image
(pre-/255) and outputs p(NSFW).

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}

SCALE = os.environ.get('BUDDY_SCALE', 'demo')   # smoke | demo | full
from tf_utils import on_gpu, tf_version
from tf_utils import set_memory_growth
set_memory_growth()
print('TF', tf_version(), '| GPU:', on_gpu(), '| scale:', SCALE)


In [ ]:
# Local NSFW image corpus (class dirs: Neutral=0, NSFW=1)
import os
import tensorflow as tf
from buddy_data import nsfw_images, reddit_nsfw

root = nsfw_images()
print('root:', root)
IMG = 224
train_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'train', image_size=(IMG, IMG), batch_size=32, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'val', image_size=(IMG, IMG), batch_size=32)
test_ds = tf.keras.utils.image_dataset_from_directory(
    root / 'test', image_size=(IMG, IMG), batch_size=32)

print('class_names:', train_ds.class_names)
nsfw_df = reddit_nsfw()
print('reddit title prior:', nsfw_df.shape, nsfw_df['is_nsfw'].value_counts().to_dict())

In [ ]:
# Balanced class weights + augmentation (NSFW is the minority class)
import numpy as np

def count_by_class(ds):
    counts = np.zeros(len(train_ds.class_names), dtype=int)
    for _, y in ds:
        counts += np.bincount(y.numpy(), minlength=len(counts))
    return counts

tr_counts = count_by_class(train_ds)
total = tr_counts.sum()
class_weight = {i: total / (len(tr_counts) * c) for i, c in enumerate(tr_counts)}
print('train counts:', dict(zip(train_ds.class_names, tr_counts.tolist())), '| weights:', class_weight)

# Bounded prefetch buffer: AUTOTUNE over-allocates on low-RAM CPU boxes and
# caused severe slow-downs; 2 batches keeps a stable ~40 MB in flight.
AUT = tf.data.AUTOTUNE
PREFETCH = 2
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])
train_ds = (train_ds
            .map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUT)
            .map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=AUT)
            .prefetch(PREFETCH))
val_ds = val_ds.map(lambda x, y: (x / 255.0, y)).prefetch(PREFETCH)
test_ds = test_ds.map(lambda x, y: (x / 255.0, y)).prefetch(PREFETCH)

if SCALE == 'smoke':                      # fast CI/verification subset
    train_ds = train_ds.take(6)
    val_ds = val_ds.take(3)
    test_ds = test_ds.take(3)


In [ ]:
# Frozen MobileNetV3Small head -> staged fine-tune
base = tf.keras.applications.MobileNetV3Small(
    weights='imagenet', include_top=False, input_shape=(IMG, IMG, 3))
base.trainable = False
x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
m = tf.keras.Model(base.input, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy',
          metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
m.summary()

In [ ]:
# Train head (frozen backbone), then fine-tune the top block (early-stopped)
# EarlyStopping + ReduceLROnPlateau make the run robust across BUDDY_SCALE:
# more epochs no longer mean overfitting, the best val_loss weights are kept.
import time
EPOCHS = {'smoke': 2, 'demo': 8, 'full': 12}[SCALE]
FT_EPOCHS = {'smoke': 0, 'demo': 3, 'full': 5}[SCALE]
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=1, min_lr=1e-6),
]
t0 = time.time()
m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
      class_weight=class_weight, callbacks=callbacks, verbose=1)

if FT_EPOCHS > 0:
    base.trainable = True
    for layer in base.layers[: int(0.7 * len(base.layers))]:
        layer.trainable = False
    m.compile(tf.keras.optimizers.Adam(1e-5), 'binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    m.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS,
          callbacks=callbacks, verbose=1)
print(f'total {time.time()-t0:.0f}s')


In [ ]:
# Evaluate on the untouched test split: AUC / P / R / threshold calibration
from sklearn.metrics import average_precision_score, precision_recall_curve

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_score = np.concatenate([m.predict(x, verbose=0)[:, 0] for x, _ in test_ds])
from sklearn.metrics import roc_auc_score
auc = float(roc_auc_score(y_true, y_score))
ap = float(average_precision_score(y_true, y_score))
prec, rec, thr = precision_recall_curve(y_true, y_score)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best = float(thr[np.argmax(f1[:-1])])
print(f'test n={len(y_true)} positive rate={y_true.mean():.3f}')
print(f'test AUC={auc:.3f} AP={ap:.3f} | F1-best threshold={best:.3f}')
print('confusion @', round(best, 2), '->',
      {'TP': int(((y_score > best) & (y_true == 1)).sum()),
       'FP': int(((y_score > best) & (y_true == 0)).sum()),
       'TN': int(((y_score <= best) & (y_true == 0)).sum()),
       'FN': int(((y_score <= best) & (y_true == 1)).sum())})
# Overfit gauge: AUC on an un-augmented train sample vs the held-out test set.
train_gauge = tf.keras.utils.image_dataset_from_directory(
    root / 'train', image_size=(IMG, IMG), batch_size=32, shuffle=True).take({'smoke': 3, 'demo': 63, 'full': 128}[SCALE])
train_gauge = train_gauge.map(lambda x, y: (x / 255.0, y)).prefetch(PREFETCH)
y_g = np.concatenate([y.numpy() for _, y in train_gauge])
y_g_score = np.concatenate([m.predict(x, verbose=0)[:, 0] for x, _ in train_gauge])
train_auc = float(roc_auc_score(y_g, y_g_score))
print(f'overfit gauge: train AUC (2000-img sample)={train_auc:.3f} | test AUC={auc:.3f} | gap={train_auc - auc:+.3f}')


### Export contract (consumed by the AI service)

The cells below write `../models/nsfw_classifier.onnx` and its dynamic-INT8 quantized copy
`nsfw_classifier_int8.onnx`. `app/ml/serving.py::load_preferred('nsfw_classifier')` loads the
`_int8.onnx` artifact from `AI_MODEL_CACHE_DIR` (dev: bind-mounted to
`backend/ai_service/models/`). The model card JSON is what the `apps.ai` Django
`ModelMetadata` sync endpoint expects.


In [ ]:
# Export ONNX (+ INT8) for app/ml/serving.py::load_preferred('nsfw_classifier')
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(m, Path('../models'), 'nsfw_classifier', '1.0.0')
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'nsfw_classifier', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'metrics': {'test_auc': float(auc), 'test_ap': float(ap), 'threshold': float(best)}})
print('exported', q)